# Watch trained agents — Pistonball & Multiwalker

Roll out **saved checkpoints** as compact looping GIFs. Each frame is stamped with the experiment, seed, physics config, live return, and the stored eval KPI (IQM / mean for that context when `evaluation_results.json` exists).

Pistonball clips drop the empty white sky so the pistons stay readable at a small size. **Every seed** of each named experiment is loaded unless you set `SEEDS`.

| `COMPARE` | Grid |
|---|---|
| `"envs"` | columns = physics configs, rows = seeds (one experiment, or several) |
| `"policies"` | one physics config; columns = seeds, rows = experiments |

Launch Jupyter from the repo root or `notebooks/`. Headless machines use a dummy pygame display automatically.

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")

import pandas as pd
import torch
from IPython.display import display

import importlib
import sys

REPO_ROOT = Path("..").resolve()
if not (REPO_ROOT / "evaluation").is_dir():
    REPO_ROOT = Path(".").resolve()
sys.path.insert(0, str(REPO_ROOT))

import evaluation.rollout_viz as _rollout_viz

importlib.reload(_rollout_viz)
from evaluation.rollout_viz import (
    list_available_runs,
    list_context_names,
    load_policy,
    resolve_run_dirs,
    show_clips,
    suggested_contexts,
    watch_specs,
)

RUNS_ROOT = REPO_ROOT / "runs"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("repo:", REPO_ROOT)
print("device:", DEVICE)

repo: /gpfs/bwfor/home/fr/fr_fr/fr_jk1376/projects/cmarl
device: cpu


## What is on disk?

Use the `preset` column (or `exp_name`) as `POLICIES` entries.

In [ ]:
runs = list_available_runs(RUNS_ROOT)
runs_df = pd.DataFrame(runs)
if runs_df.empty:
    print(f"No checkpointed runs under {RUNS_ROOT}")
else:
    cols = ["group", "preset", "label", "seed", "family", "method_tag", "checkpoint", "finished"]
    display(runs_df[[c for c in cols if c in runs_df.columns]])
    print(
        "TensorBoard by family:\n"
        "  tensorboard --logdir runs/pistonball/n5\n"
        "  tensorboard --logdir runs/pistonball/n8\n"
        "  tensorboard --logdir runs/multiwalker\n"
        "  tensorboard --logdir_spec=n5:runs/pistonball/n5,n8:runs/pistonball/n8,mw:runs/multiwalker"
    )


## What to watch

- `POLICIES`: preset keys, experiment names, or run-folder names. **All seeds** are rolled out unless `SEEDS` is a list.
- `CONTEXTS`: eval-point names such as `default`, `ball_mass_x2`, `gravity_x0.5`.
- HUD on each frame: `experiment s{seed}` / `context split  R{live}  eval{IQM}`.

Keep `MAX_WIDTH` small (~160–200) so a full seed row fits on screen. Pistonball GIFs are cropped to the playfield automatically.

In [11]:
# "envs" = columns are physics configs (rows = seeds)
# "policies" = one physics config (rows = experiments, columns = seeds)
COMPARE = "envs"

# Copy `preset` or `exp_name` from the table above. Live Multiwalker pilots:
POLICIES = ["mw3_hidden_varying_fwd5", "mw3_concat_varying_fwd5"]
# POLICIES = ["mw3_concat_varying_fwd5_2m", "mw3_concat_varying_fwd5_tp0.1"]
# Archived fwd=1 / 200k smokes live under old_runs/ (set RUNS_ROOT accordingly):
# POLICIES = ["mw3_hidden_varying", "mw3_concat_varying"]

# CONTEXTS = ["default", "ball_mass_x0.5", "ball_mass_x2"]
CONTEXTS = ["default", "gravity_x0.5", "gravity_x2", "package_mass_x2"]
# CONTEXTS = ["default", "elasticity_x0.5", "elasticity_x2"]

SEEDS = None          # all seeds; or e.g. [1, 2, 3]
CHECKPOINT = "best"
EPISODE_SEED = 0      # env reset seed (same across panels)
MAX_FRAMES = None     # e.g. 200 to cap long Multiwalker episodes
FPS = 12
MAX_WIDTH = 176       # px; smaller => more clips per row
CROP_SKY = True       # Pistonball only: drop empty white sky
OVERLAY = True        # stamp experiment / context / KPI on every frame


## Load run list

Checkpoints are loaded one seed at a time during rollout (not all at once).

In [ ]:
if COMPARE == "envs":
    policy_specs = list(POLICIES)
    contexts = list(CONTEXTS)
elif COMPARE == "policies":
    policy_specs = list(POLICIES)
    contexts = [CONTEXTS[0]]
else:
    raise ValueError("COMPARE must be 'envs' or 'policies'")

run_dirs = [d for spec in policy_specs for d in resolve_run_dirs(RUNS_ROOT, spec, seeds=SEEDS)]
if not run_dirs:
    raise FileNotFoundError(f"no runs for {policy_specs} under {RUNS_ROOT}")

print(f"{len(run_dirs)} run(s) × {len(contexts)} context(s) = {len(run_dirs) * len(contexts)} clips")
for d in run_dirs:
    print(" ", d.name)

probe = load_policy(run_dirs[0], checkpoint=CHECKPOINT, device=DEVICE)
print("\nSuggested configs:", suggested_contexts(probe.env_kind))
print("All eval-grid names:")
print(", ".join(list_context_names(probe)))
del probe

## Roll out and compare

Same `EPISODE_SEED` is used across panels so spawns stay comparable. `eval` on the HUD is that seed's stored IQM (or mean) for this context — the number the policy is supposed to get — next to the live episode return `R`.

In [ ]:
print(f"{policy_specs}  →  {contexts}")
clips = watch_specs(
    RUNS_ROOT,
    policy_specs,
    contexts,
    seeds=SEEDS,
    checkpoint=CHECKPOINT,
    device=DEVICE,
    episode_seed=EPISODE_SEED,
    max_frames=MAX_FRAMES,
)
for clip in clips:
    print(f"{clip.title:48s}  {clip.caption}")

show_clips(
    clips,
    fps=FPS,
    max_width=MAX_WIDTH,
    crop=CROP_SKY,
    overlay=OVERLAY,
)